In [ ]:
# Project root setup
import os
import sys
from pathlib import Path

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Could not locate the project root containing src/.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [1]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ.setdefault("PYTHONWARNINGS", "ignore::FutureWarning:sklearn.linear_model._base")

import signal
import numpy as np
import pandas as pd
from numpy import linalg as la
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import f1_score
import cvxpy as cp
import random
import time
from datetime import datetime
from joblib import Parallel, delayed

from src.model import Nonneg_dagma, MetMulDagma, MetMulColide
import src.utils as utils

from baselines.colide import colide_ev, colide_nv
from baselines.dagma_linear import DAGMA_linear
from baselines.nonnegative_dagma_linear import NonnegativeDAGMA_linear
from baselines.golem import GOLEM_EV, GOLEM_TF_EV, GOLEM_TF_NV
from baselines.nofears import NoFearsLinear
from baselines.notears import notears_linear
from baselines.sortnregress import VarSortNRegress, R2SortNRegress, RandomSortNRegress
from baselines.daguerreotype import DAGuerreotype

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module=r"sklearn\.linear_model\._base")

SEED = 10
N_CPUS = max(1, int(os.environ.get("N_CPUS", os.cpu_count() or 1)))
JOBLIB_VERBOSE = max(0, int(os.environ.get("JOBLIB_VERBOSE", 10)))
SELECTED_SCENARIOS = {
    scenario.strip()
    for scenario in os.environ.get("SCENARIOS", "").split(",")
    if scenario.strip()
}
LOAD = True
SAVE_RESULTS = False
RESULTS_PATH = './results/preliminary'
os.makedirs(RESULTS_PATH, exist_ok=True)

np.random.seed(SEED)


def log_status(message):
    timestamp = datetime.now().isoformat(timespec='seconds')
    print(f'[{timestamp} pid={os.getpid()}] {message}', flush=True)


def _handle_termination(signum, frame):
    raise KeyboardInterrupt(f"Received signal {signum}; stopping experiments")


signal.signal(signal.SIGTERM, _handle_termination)
random.seed(SEED)


/home/srey/Investigacion/cvx_dag_learning/.venv-dag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
Exps = [
    # Simple Gradient Descent
    # {'model': Nonneg_dagma, 'args': {'stepsize': 5e-3, 'alpha': 2, 's': 1, 'lamb': 1e-1, 'max_iters': 1000000, 'tol': 1e-6},
    #  'init': {'acyclicity': 'logdet', 'primal_opt': 'pgd'}, 'standarize': False, 'fix_lamb': False, 'leg': 'PGD'},

    # {'model': Nonneg_dagma, 'args': {'stepsize': 5e-3, 'alpha': 2, 's': 1, 'lamb': 1e-1, 'max_iters': 1000000, 'tol': 1e-6},
    #  'init': {'acyclicity': 'logdet', 'primal_opt': 'adam'}, 'standarize': False, 'fix_lamb': False, 'leg': 'PGD-Adam'},
    
    # NOMAD
    {'model': MetMulDagma, 'args': {'stepsize': 3e-4, 'alpha_0': .01, 'rho_0': .05, 's': 1, 'lamb': .1, 'iters_in': 10000, 'step_type': 'fixed',
     'iters_out': 10, 'beta': 2}, 'init': {'acyclicity': 'logdet', 'primal_opt': 'adam'}, 'standarize': False,
     'fix_lamb': False, 'leg': 'NOMAD-adam'},

    {'model': MetMulDagma, 'args': {'stepsize': 1e-5, 'alpha_0': .01, 'rho_0': .01, 's': 1, 'lamb': .2, 'iters_in': 5000, 'step_type': 'fixed',
     'iters_out': 50, 'beta': 1.5}, 'init': {'acyclicity': 'logdet', 'primal_opt': 'fista', 'restart': True}, 'standarize': False,
     'fix_lamb': False, 'leg': 'NOMAD-fista'},


    # # COLIDE - NOMAD
    # {'model': MetMulColide, 'args': {'stepsize': 5e-5, 'alpha_0': .1, 'rho_0': .1, 's': 1, 'lamb': .1, 'iters_in': 30000,
    #  'iters_out': 10, 'beta': 1.2}, 'init': {'acyclicity': 'logdet', 'primal_opt': 'fista', 'restart': True}, 'standarize': False,
    #  'fix_lamb': False, 'leg': 'MM-Col-fista-r'},

    # {'model': MetMulColide, 'args': {'stepsize': 5e-5, 'alpha_0': .1, 'rho_0': .1, 's': 1, 'lamb': .1, 'iters_in': 30000,
    #  'iters_out': 10, 'beta': 1.2}, 'init': {'acyclicity': 'logdet', 'primal_opt': 'adam', 'restart': True}, 'standarize': False,
    #  'fix_lamb': False, 'leg': 'MM-Col-adam'},

    # {'model': MetMulColide, 'args': {'stepsize': 3e-4, 'alpha_0': .01, 'rho_0': .05, 's': 1, 'lamb': .1, 'iters_in': 20000,
    #  'iters_out': 10, 'beta': 2, 'sca_adam': True}, 'init': {'acyclicity': 'logdet', 'primal_opt': 'sca'}, 'standarize': False,
    #  'fix_lamb': False, 'leg': 'MM-Col-sca'},

    ### BASELINES ###
    # NoTears
    {'model': notears_linear, 'args': {'loss_type': 'l2', 'lambda1': .1, 'max_iter': 10}, 'standarize': False, 'leg': 'NoTears'},

    # NoFears: NOTEARS followed by the official KKTS local search
    {'model': NoFearsLinear, 'args': {'lambda1': .1, 'w_threshold': .3, 'max_iter': 100, 'h_tol': 1e-10,
     'rho_init': 1., 'rho_factor': 10., 'rho_max': 1e16, 'h_progress_rate': .25, 'w_tol': 1e-10,
     'pen_tol': 0., 'rev_edges': 'alt-full', 'minimize_z': True, 'init_no_pen': True, 'no_pen': False},
     'standarize': False, 'fix_lamb': True, 'leg': 'NoFears'},
    
    # DAGMA
    {'model': DAGMA_linear, 'init': {'loss_type': 'l2'}, 'args': {'lambda1': .05, 'T': 4, 's': [1.0, .9, .8, .7],
     'warm_iter': 2e4, 'max_iter': 7e4, 'lr': .0003}, 'standarize': False, 'leg': 'DAGMA'},

    {'model': NonnegativeDAGMA_linear, 'init': {'loss_type': 'l2'}, 'args': {'lambda1': .05, 'T': 4, 's': [1.0, .9, .8, .7],
     'warm_iter': 2e4, 'max_iter': 7e4, 'lr': .0003}, 'standarize': False, 'leg': 'NonDAGMA'},

    # Colide
    {'model': colide_ev, 'args': {'lambda1': .05, 'T': 4, 's': [1.0, .9, .8, .7], 'warm_iter': 2e4,
     'max_iter': 7e4, 'lr': .0003}, 'standarize': False, 'leg': 'CoLiDE-EV'},
    
    {'model': colide_nv, 'args': {'lambda1': .05, 'T': 4, 's': [1.0, .9, .8, .7], 'warm_iter': 2e4,
     'max_iter': 7e4, 'lr': .0003}, 'standarize': False, 'leg': 'CoLiDE-NV'},
     
    # GOLEM
    {'model': GOLEM_EV, 'args': {'lambda1': 2e-2, 'lambda2': 5.0, 'num_iter': 100000, 'learning_rate': 1e-3, 'w_threshold': 0.3,
     'postprocess': True, 'checkpoint': None}, 'standarize': False, 'fix_lamb': True, 'leg': 'GOLEM-EV-np'},

    {'model': GOLEM_TF_EV, 'args': {'lambda1': 2e-2, 'lambda2': 5.0, 'num_iter': 100000, 'learning_rate': 1e-3, 'w_threshold': 0.3,
     'postprocess': True, 'checkpoint': None}, 'standarize': False, 'fix_lamb': True, 'leg': 'GOLEM-EV'},
     
    {'model': GOLEM_TF_NV, 'init': {'init_with_ev': True}, 'args': {'lambda1': 2e-3, 'lambda2': 5.0, 'lambda1_ev': 2e-2, 'lambda2_ev': 5.0,
     'num_iter': 100000, 'num_iter_ev': 100000, 'learning_rate': 1e-3, 'learning_rate_ev': 1e-3, 'w_threshold': 0.3, 'postprocess': True,
     'checkpoint': None}, 'standarize': False, 'fix_lamb': True, 'leg': 'GOLEM-NV'},

    # Regress
    {'model': VarSortNRegress, 'args': {'w_threshold': 0.3}, 'standarize': False, 'fix_lamb': True, 'leg': 'SortNRegress'},

    # {'model': R2SortNRegress, 'args': {}, 'standarize': False, 'fix_lamb': True, 'leg': 'R2-SortNRegress'},

    # # DAGuerreotype
    # {'model': DAGuerreotype, 'init': {'structure': 'sp_map', 'sparsifier': 'l0_ber_ste', 'equations': 'linear', 'loss': 'nll_ev',
    #  'joint': False, 'nogpu': True, 'standardize': False, 'init_theta': 'zeros', 'num_epochs': 5000, 'num_inner_iters': 200, 'lr': 1e-1,
    #  'lr_theta': 1e-1, 'pruning_reg': 0.001, 'l2_theta': 0.0005, 'l2_eq': 0.0005}, 'args': {}, 'standarize': False, 'fix_lamb': True, 'leg': 'DAGuerreotype'}
]

# Keep model definitions and hyperparameters identical; only the input scale changes.
Exps_standardized = [{**exp, 'standarize': True} for exp in Exps]


In [3]:

def get_lamb_value(n_nodes, n_samples, times=1):
    return np.sqrt(np.log(n_nodes) / n_samples) * times 


def run_parallel_exps(data_p, exps, n_dags, scenario_name, thr=.3, verb=False):
    n_jobs = max(1, min(N_CPUS, n_dags))
    log_status(
        f'RUN scenario={scenario_name} dags={n_dags} workers={n_jobs} '
        f'joblib_verbose={JOBLIB_VERBOSE}'
    )
    t_init = time.time()

    parallel = Parallel(n_jobs=n_jobs, verbose=JOBLIB_VERBOSE)
    try:
        results = parallel(
            delayed(run_exps)(g, data_p, exps, thr=thr, verb=verb)
            for g in range(n_dags)
        )
    except (KeyboardInterrupt, SystemExit):
        backend = getattr(parallel, "_backend", None)
        if backend is not None and hasattr(backend, "terminate"):
            backend.terminate()
        raise
    finally:
        backend = getattr(parallel, "_backend", None)
        if backend is not None and hasattr(backend, "terminate"):
            backend.terminate()

    elapsed_time = (time.time() - t_init)/60
    log_status(f'DONE scenario={scenario_name} elapsed_minutes={elapsed_time:.3f}')
    return results

def run_exps(g, data_p, exps, thr=.3, verb=False):
    A_true, _, X = utils.simulate_sem(**data_p)
    A_true_bin = utils.to_bin(A_true, thr)
    X_std = utils.standarize(X)
    X_scale = X.std(axis=0)
    A_true_std = A_true * X_scale[:, None] / X_scale[None, :]

    M, N = X.shape

    Z = X - X @ A_true
    fidelity = 1/data_p['n_samples']*la.norm(Z, 'fro')**2
    fidelity_std = 1/data_p['n_samples']*la.norm(X_std - X_std @ A_true_std, 'fro')**2

    Sigma_hat = la.norm(Z, axis=0) / np.sqrt(M)

    log_status(
        f'DAG {g}: fidelity={fidelity:.3f} fidelity_std={fidelity_std:.3f} '
        f'sigma_min={np.min(Sigma_hat):.4f} sigma_max={np.max(Sigma_hat):.4f}'
    )

    shd, fscore, sid_norm, err, acyc, runtime = [np.zeros(len(exps))  for _ in range(6)]
    for i, exp in enumerate(exps):
        standardized = exp.get('standarize', False)
        X_aux = X_std if standardized else X
        A_true_aux = A_true_std if standardized else A_true

        args = exp['args'].copy()
        if 'fix_lamb' in exp.keys() and not exp['fix_lamb']:
            args['lamb'] = get_lamb_value(N, M, args['lamb'])

        if 'know_var' in exp.keys() and exp['know_var']:
            args['Sigma'] = data_p['var']

        model = None
        try:
            if exp['model'] == notears_linear:
                t_i = time.time()
                A_est = notears_linear(X_aux, **args)
                t_solved = time.time() - t_i
            else:
                model = exp['model'](**exp['init']) if 'init' in exp.keys() else exp['model']()
                t_i = time.time()
                model.fit(X_aux, **args)
                t_solved = time.time() - t_i

                A_est = model.W_est
                if getattr(model, 'cycle_repair_applied_', False):
                    log_status(
                        f'DAG {g} method={exp["leg"]} exact-DAG safeguard removed '
                        f'{model.cycle_repair_count_} edge(s): '
                        f'{model.cycle_repair_edges_}'
                    )
        except Exception as exc:
            raise RuntimeError(
                f'DAG {g} method={exp["leg"]} failed: '
                f'{type(exc).__name__}: {exc}'
            ) from exc

        A_est_bin = utils.to_bin(A_est, thr)
        try:
            shd[i], _, _, sid_norm[i] = utils.count_accuracy(
                A_true_bin,
                A_est_bin,
                compute_sid=True,
                sid_normalize=True,
            )
        except ValueError:
            shd[i], _, _ = utils.count_accuracy(A_true_bin, A_est_bin)
            sid_norm[i] = np.nan
        fscore[i] = f1_score(A_true_bin.flatten(), A_est_bin.flatten())
        err[i] = utils.compute_norm_sq_err(A_true_aux, A_est)
        acyc[i] = model.dagness(A_est) if model is not None and hasattr(model, 'dagness') else float(not utils.is_dag(A_est_bin))
        runtime[i] = t_solved

        if verb and (g % N_CPUS == 0):
            sid_text = f'{sid_norm[i]:.3f}' if np.isfinite(sid_norm[i]) else 'nan'
            log_status(
                f'DAG {g} method={exp["leg"]} standardized={standardized} '
                f'shd={shd[i]} sid={sid_text} err={err[i]:.3f} time={runtime[i]:.3f}'
            )

    return shd, fscore, sid_norm, err, acyc, runtime

def preliminary_results_prefix(scenario_name):
    return f'{RESULTS_PATH}/preliminary_{scenario_name}'


def load_preliminary_results(scenario_name):
    tables = {}
    exps_leg = None
    for agg in ('mean', 'median'):
        file_name = f'{preliminary_results_prefix(scenario_name)}_{agg}.csv'
        if not os.path.exists(file_name):
            raise FileNotFoundError(f'Results file not found: {file_name}')
        tables[agg] = pd.read_csv(file_name)
        if 'leg' not in tables[agg].columns:
            raise ValueError(f'Results file has no experiment legend column: {file_name}')

        table_exps_leg = tables[agg]['leg'].astype(str).tolist()
        if exps_leg is None:
            exps_leg = table_exps_leg
        elif table_exps_leg != exps_leg:
            raise ValueError(f'Experiment legends differ between saved tables for {scenario_name}')

        print(f'Loaded {agg} results from {file_name}')
        display(tables[agg])
    return tables, exps_leg


def run_or_load_preliminary_results(data_p, exps, n_dags, scenario_name, thr=.3, verb=False):
    if SELECTED_SCENARIOS and scenario_name not in SELECTED_SCENARIOS:
        log_status(f'SKIP scenario={scenario_name} selected={sorted(SELECTED_SCENARIOS)}')
        return None, None, None

    standardized_count = sum(exp.get('standarize', False) for exp in exps)
    mode = 'load' if LOAD else 'run'
    log_status(
        f'START scenario={scenario_name} mode={mode} nodes={data_p["n_nodes"]} '
        f'samples={data_p["n_samples"]} edges={data_p["edges"]} dags={n_dags} '
        f'standardized_methods={standardized_count}/{len(exps)}'
    )

    if LOAD:
        tables, exps_leg = load_preliminary_results(scenario_name)
        log_status(f'LOADED scenario={scenario_name} methods={len(exps_leg)}')
        return None, tables, exps_leg

    results = run_parallel_exps(
        data_p,
        exps,
        n_dags,
        scenario_name=scenario_name,
        thr=thr,
        verb=verb,
    )
    shd, fscore, sid_norm, err, acyc, runtime = zip(*results)
    metrics = {'shd': shd, 'fscore': fscore, 'sid_norm': sid_norm, 'err': err, 'acyc': acyc, 'time': runtime}

    exps_leg = [exp['leg'] for exp in exps]
    file_prefix = preliminary_results_prefix(scenario_name) if SAVE_RESULTS else None
    utils.display_results(exps_leg, metrics, agg='mean', file_name=f'{file_prefix}_mean' if file_prefix else None)
    utils.display_results(exps_leg, metrics, agg='median', file_name=f'{file_prefix}_median' if file_prefix else None)
    log_status(f'SAVED scenario={scenario_name} prefix={file_prefix}')
    return metrics, None, exps_leg



## CASE 1 - N=50, Homocedastic

In [4]:
N = 50
SCENARIO_NAME = 'er4_N50_var1'

n_dags = 50
verb = True
data_params = {
    'n_nodes': N,
    'n_samples': 1000, # 1000,
    'graph_type': 'er',
    'edges': 4*N,
    'edge_type': 'positive',
    'w_range': (.5, 1),  # (.5, 1)
    'var': 1
}
metrics, tables, exps_leg = run_or_load_preliminary_results(data_params, Exps, n_dags, SCENARIO_NAME, thr=.3, verb=verb)


[2026-06-15T11:23:10 pid=3422037] START scenario=er4_N50_var1 mode=load nodes=50 samples=1000 edges=200 dags=50 standardized_methods=0/12
Loaded mean results from ./results/preliminary/preliminary_er4_N50_var1_mean.csv


,leg,shd,fscore,sid_norm,err,acyc,time
0,NOMAD-adam,0.0200 ± 0.1400,0.9999 ± 0.0006,0.0240 ± 0.1680,0.0045 ± 0.0010,0.0020 ± 0.0007,6.7220 ± 1.0126
1,NOMAD-fista,0.1800 ± 0.5896,0.9993 ± 0.0023,0.1352 ± 0.5205,0.0057 ± 0.0044,0.0000 ± 0.0000,8.1888 ± 0.6102
2,NOMAD-adam-alpha2,9.2400 ± 5.6482,0.9718 ± 0.0169,2.6588 ± 1.8918,0.0373 ± 0.0213,0.0000 ± 0.0000,4.3660 ± 0.8500
3,NOMAD-fista-alpha2,45.4400 ± 39.2246,0.8691 ± 0.1114,6.1995 ± 4.9633,0.1994 ± 0.1696,0.0000 ± 0.0000,1.1095 ± 0.7850
4,NoTears,55.0000 ± 20.4587,0.8411 ± 0.0537,8.9256 ± 2.0014,0.2871 ± 0.0981,0.0000 ± 0.0000,120.0343 ± 31.6131
5,NoFears,43.4600 ± 23.2785,0.8782 ± 0.0620,5.9416 ± 2.8429,0.2404 ± 0.1339,0.0000 ± 0.0000,144.1568 ± 45.3105
6,DAGMA,27.6000 ± 13.6176,0.9201 ± 0.0358,7.1048 ± 2.2015,0.1305 ± 0.0547,0.0002 ± 0.0000,8.8076 ± 2.4354
7,NonDAGMA,2.1600 ± 1.9012,0.9944 ± 0.0050,0.9732 ± 0.9793,0.0166 ± 0.0079,0.0001 ± 0.0000,11.0394 ± 2.9150
8,CoLiDE-EV,26.3000 ± 13.8827,0.9244 ± 0.0378,6.6900 ± 2.5098,0.1248 ± 0.0613,0.0002 ± 0.0000,10.4823 ± 2.9191
9,CoLiDE-NV,32.9000 ± 14.7231,0.9056 ± 0.0403,7.6448 ± 2.4066,0.1528 ± 0.0650,0.0000 ± 0.0000,10.9606 ± 2.9885


Loaded median results from ./results/preliminary/preliminary_er4_N50_var1_median.csv


,leg,shd,fscore,sid_norm,err,acyc,time
0,NOMAD-adam,0.0000 ± 0.1400,1.0000 ± 0.0006,0.0000 ± 0.1680,0.0043 ± 0.0010,0.0019 ± 0.0007,6.2712 ± 1.0126
1,NOMAD-fista,0.0000 ± 0.5896,1.0000 ± 0.0023,0.0000 ± 0.5205,0.0046 ± 0.0044,0.0000 ± 0.0000,8.0931 ± 0.6102
2,NOMAD-adam-alpha2,8.0000 ± 5.6482,0.9763 ± 0.0169,1.8900 ± 1.8918,0.0320 ± 0.0213,0.0000 ± 0.0000,4.0624 ± 0.8500
3,NOMAD-fista-alpha2,69.5000 ± 39.2246,0.7961 ± 0.1114,4.9100 ± 4.9633,0.3266 ± 0.1696,0.0000 ± 0.0000,0.5459 ± 0.7850
4,NoTears,52.5000 ± 20.4587,0.8458 ± 0.0537,9.0600 ± 2.0014,0.2705 ± 0.0981,0.0000 ± 0.0000,120.1966 ± 31.6131
5,NoFears,42.0000 ± 23.2785,0.8798 ± 0.0620,6.0200 ± 2.8429,0.2271 ± 0.1339,0.0000 ± 0.0000,141.2395 ± 45.3105
6,DAGMA,24.5000 ± 13.6176,0.9300 ± 0.0358,7.0100 ± 2.2015,0.1178 ± 0.0547,0.0002 ± 0.0000,8.4371 ± 2.4354
7,NonDAGMA,2.0000 ± 1.9012,0.9951 ± 0.0050,0.7600 ± 0.9793,0.0145 ± 0.0079,0.0001 ± 0.0000,10.7373 ± 2.9150
8,CoLiDE-EV,23.0000 ± 13.8827,0.9329 ± 0.0378,6.6300 ± 2.5098,0.1178 ± 0.0613,0.0002 ± 0.0000,9.9496 ± 2.9191
9,CoLiDE-NV,32.0000 ± 14.7231,0.9067 ± 0.0403,7.7900 ± 2.4066,0.1494 ± 0.0650,0.0000 ± 0.0000,10.4727 ± 2.9885


[2026-06-15T11:23:10 pid=3422037] LOADED scenario=er4_N50_var1 methods=13


In [5]:
# Results are displayed by run_or_load_preliminary_results.


## CASE 2 - N=100, Homocedastic

In [6]:
N = 100
SCENARIO_NAME = 'er4_N100_var1'

n_dags = 50
verb = True
data_params = {
    'n_nodes': N,
    'n_samples': 1000, # 1000,
    'graph_type': 'er',
    'edges': 4*N,
    'edge_type': 'positive',
    'w_range': (.5, 1),  # (.5, 1)
    'var': 1
}
metrics, tables, exps_leg = run_or_load_preliminary_results(data_params, Exps, n_dags, SCENARIO_NAME, thr=.3, verb=verb)


[2026-06-15T11:23:10 pid=3422037] START scenario=er4_N100_var1 mode=load nodes=100 samples=1000 edges=400 dags=50 standardized_methods=0/12
Loaded mean results from ./results/preliminary/preliminary_er4_N100_var1_mean.csv


,leg,shd,fscore,sid_norm,err,acyc,time
0,NOMAD-adam,0.0000 ± 0.0000,1.0000 ± 0.0000,0.0000 ± 0.0000,0.0050 ± 0.0003,0.0071 ± 0.0085,23.4055 ± 3.0983
1,NOMAD-fista,0.1600 ± 0.5044,0.9997 ± 0.0009,0.1352 ± 0.4074,0.0045 ± 0.0019,0.0000 ± 0.0000,19.8523 ± 1.5077
2,NOMAD-adam-alpha2,19.2200 ± 8.8978,0.9713 ± 0.0130,3.9728 ± 2.1607,0.0360 ± 0.0150,0.0001 ± 0.0001,13.5625 ± 3.3192
3,NOMAD-fista-alpha2,54.5000 ± 59.8389,0.9192 ± 0.0869,7.9073 ± 6.7287,0.1303 ± 0.1440,0.0000 ± 0.0000,3.8447 ± 2.1879
4,NoTears,44.5800 ± 18.6409,0.9317 ± 0.0245,10.6394 ± 3.0897,0.1197 ± 0.0432,0.0000 ± 0.0000,1085.5689 ± 407.0932
5,NoFears,30.7000 ± 16.1273,0.9551 ± 0.0221,5.8588 ± 2.9790,0.0829 ± 0.0387,0.0000 ± 0.0000,1135.7218 ± 313.7918
6,DAGMA,21.8600 ± 11.2125,0.9651 ± 0.0152,7.3726 ± 2.1631,0.0610 ± 0.0240,0.0002 ± 0.0000,24.6124 ± 6.7821
7,NonDAGMA,1.1400 ± 1.2330,0.9981 ± 0.0022,0.7714 ± 0.9178,0.0079 ± 0.0029,0.0001 ± 0.0001,28.9907 ± 5.8322
8,CoLiDE-EV,22.7200 ± 10.2509,0.9639 ± 0.0147,7.3960 ± 2.1651,0.0610 ± 0.0229,0.0002 ± 0.0000,31.2113 ± 8.4399
9,CoLiDE-NV,27.0200 ± 12.5722,0.9572 ± 0.0183,8.1032 ± 2.3820,0.0692 ± 0.0264,0.0000 ± 0.0000,33.7270 ± 10.4179


Loaded median results from ./results/preliminary/preliminary_er4_N100_var1_median.csv


,leg,shd,fscore,sid_norm,err,acyc,time
0,NOMAD-adam,0.0000 ± 0.0000,1.0000 ± 0.0000,0.0000 ± 0.0000,0.0050 ± 0.0003,0.0041 ± 0.0085,22.6063 ± 3.0983
1,NOMAD-fista,0.0000 ± 0.5044,1.0000 ± 0.0009,0.0000 ± 0.4074,0.0041 ± 0.0019,0.0000 ± 0.0000,19.8820 ± 1.5077
2,NOMAD-adam-alpha2,18.0000 ± 8.8978,0.9720 ± 0.0130,3.5400 ± 2.1607,0.0345 ± 0.0150,0.0001 ± 0.0001,11.6181 ± 3.3192
3,NOMAD-fista-alpha2,14.0000 ± 59.8389,0.9788 ± 0.0869,4.5400 ± 6.7287,0.0301 ± 0.1440,0.0000 ± 0.0000,5.2297 ± 2.1879
4,NoTears,39.0000 ± 18.6409,0.9391 ± 0.0245,9.9550 ± 3.0897,0.1127 ± 0.0432,0.0000 ± 0.0000,1009.2120 ± 407.0932
5,NoFears,28.5000 ± 16.1273,0.9593 ± 0.0221,5.6450 ± 2.9790,0.0810 ± 0.0387,0.0000 ± 0.0000,1161.3455 ± 313.7918
6,DAGMA,20.0000 ± 11.2125,0.9678 ± 0.0152,7.1250 ± 2.1631,0.0554 ± 0.0240,0.0002 ± 0.0000,24.1138 ± 6.7821
7,NonDAGMA,1.0000 ± 1.2330,0.9987 ± 0.0022,0.4900 ± 0.9178,0.0074 ± 0.0029,0.0001 ± 0.0001,29.2524 ± 5.8322
8,CoLiDE-EV,21.0000 ± 10.2509,0.9670 ± 0.0147,7.5350 ± 2.1651,0.0549 ± 0.0229,0.0002 ± 0.0000,29.9355 ± 8.4399
9,CoLiDE-NV,24.5000 ± 12.5722,0.9604 ± 0.0183,7.9250 ± 2.3820,0.0617 ± 0.0264,0.0000 ± 0.0000,32.7307 ± 10.4179


[2026-06-15T11:23:10 pid=3422037] LOADED scenario=er4_N100_var1 methods=13


In [7]:
# Results are displayed by run_or_load_preliminary_results.


## CASE 3 - N=200, Weights - [0.5, 2]

In [8]:
N = 200
SCENARIO_NAME = 'er4_N200_var1'

n_dags = 50
verb = True
data_params = {
    'n_nodes': N,
    'n_samples': 1000, # 1000,
    'graph_type': 'er',
    'edges': 4*N,
    'edge_type': 'positive',
    'w_range': (.5, 2),  # (.5, 1)
    'var': 1
}
metrics, tables, exps_leg = run_or_load_preliminary_results(data_params, Exps, n_dags, SCENARIO_NAME, thr=.3, verb=verb)


[2026-06-15T11:23:10 pid=3422037] START scenario=er4_N200_var1 mode=load nodes=200 samples=1000 edges=800 dags=50 standardized_methods=0/12
Loaded mean results from ./results/preliminary/preliminary_er4_N200_var1_mean.csv


,leg,shd,fscore,sid_norm,err,acyc,time
0,NOMAD-adam,151.1000 ± 217.1526,0.9244 ± 0.0897,0.3557 ± 0.9740,0.0648 ± 0.1000,0.1123 ± 0.1890,276.7458 ± 29.6651
1,NOMAD-fista,692.2000 ± 147.1663,0.4441 ± 0.1731,35.8418 ± 5.9133,0.8477 ± 0.2897,0.0002 ± 0.0010,139.6002 ± 57.1149
2,NOMAD-adam-alpha2,141.4600 ± 182.8382,0.9239 ± 0.0768,3.7588 ± 2.0770,0.0644 ± 0.0873,0.0599 ± 0.1270,266.7587 ± 47.9282
3,NOMAD-fista-alpha2,708.1000 ± 113.1138,0.4362 ± 0.1496,35.7201 ± 5.8898,0.8585 ± 0.2575,0.0002 ± 0.0008,134.9045 ± 61.6943
4,NoTears,456.1800 ± 150.8298,0.6875 ± 0.1282,27.3994 ± 7.3005,0.4799 ± 0.2391,0.0200 ± 0.1400,25811.7790 ± 5053.5606
5,NoFears,304.0600 ± 214.6011,0.8084 ± 0.1276,20.3515 ± 11.8718,0.3342 ± 0.2407,0.0000 ± 0.0000,27645.9760 ± 4028.0109
6,DAGMA,326.1200 ± 226.5701,0.8208 ± 0.0978,14.3253 ± 4.5474,0.2221 ± 0.1491,0.0029 ± 0.0043,568.5031 ± 158.8032
7,NonDAGMA,1683.2000 ± 142.9856,0.2732 ± 0.0353,nan ± nan,1.2906 ± 0.0808,1.0346 ± 0.2998,25.9182 ± 9.8726
8,CoLiDE-EV,307.4600 ± 207.8846,0.8289 ± 0.0935,14.1149 ± 4.6132,0.2097 ± 0.1422,0.0026 ± 0.0035,859.4114 ± 208.1736
9,CoLiDE-NV,298.7400 ± 186.9617,0.8281 ± 0.0879,16.1635 ± 4.6323,0.2185 ± 0.1414,0.0000 ± 0.0000,962.0173 ± 222.2420


Loaded median results from ./results/preliminary/preliminary_er4_N200_var1_median.csv


,leg,shd,fscore,sid_norm,err,acyc,time
0,NOMAD-adam,72.0000 ± 217.1526,0.9577 ± 0.0897,0.0000 ± 0.9740,0.0284 ± 0.1000,0.0372 ± 0.1890,288.4179 ± 29.6651
1,NOMAD-fista,741.5000 ± 147.1663,0.4064 ± 0.1731,37.2600 ± 5.9133,0.9019 ± 0.2897,0.0000 ± 0.0010,117.9868 ± 57.1149
2,NOMAD-adam-alpha2,87.0000 ± 182.8382,0.9473 ± 0.0768,3.5200 ± 2.0770,0.0388 ± 0.0873,0.0007 ± 0.1270,287.2822 ± 47.9282
3,NOMAD-fista-alpha2,742.0000 ± 113.1138,0.4094 ± 0.1496,37.0950 ± 5.8898,0.9106 ± 0.2575,-0.0000 ± 0.0008,122.7869 ± 61.6943
4,NoTears,427.0000 ± 150.8298,0.7225 ± 0.1282,25.6250 ± 7.3005,0.4104 ± 0.2391,0.0000 ± 0.1400,25783.7648 ± 5053.5606
5,NoFears,251.5000 ± 214.6011,0.8387 ± 0.1276,16.9250 ± 11.8718,0.2540 ± 0.2407,0.0000 ± 0.0000,28441.6980 ± 4028.0109
6,DAGMA,253.0000 ± 226.5701,0.8418 ± 0.0978,13.6300 ± 4.5474,0.1837 ± 0.1491,0.0015 ± 0.0043,524.3941 ± 158.8032
7,NonDAGMA,1662.5000 ± 142.9856,0.2789 ± 0.0353,nan ± nan,1.2711 ± 0.0808,1.0649 ± 0.2998,24.1890 ± 9.8726
8,CoLiDE-EV,260.0000 ± 207.8846,0.8456 ± 0.0935,13.4425 ± 4.6132,0.1748 ± 0.1422,0.0014 ± 0.0035,844.6906 ± 208.1736
9,CoLiDE-NV,244.5000 ± 186.9617,0.8451 ± 0.0879,15.6550 ± 4.6323,0.1795 ± 0.1414,0.0000 ± 0.0000,939.7830 ± 222.2420


[2026-06-15T11:23:10 pid=3422037] LOADED scenario=er4_N200_var1 methods=13


In [9]:
# Results are displayed by run_or_load_preliminary_results.


## CASE 4 - N=100, Heterocedastic

In [10]:
N = 100
SCENARIO_NAME = 'er4_N100_hetero_var'

var = np.random.uniform(low=0.5, high=5.0, size=N)

n_dags = 50
verb = True
data_params = {
    'n_nodes': N,
    'n_samples': 1000, # 1000,
    'graph_type': 'er',
    'edges': 4*N,
    'edge_type': 'positive',
    'w_range': (.5, 1),  # (.5, 1)
    'var': var**2
}
metrics, tables, exps_leg = run_or_load_preliminary_results(data_params, Exps, n_dags, SCENARIO_NAME, thr=.3, verb=verb)


[2026-06-15T11:23:10 pid=3422037] START scenario=er4_N100_hetero_var mode=load nodes=100 samples=1000 edges=400 dags=50 standardized_methods=0/12


FileNotFoundError: Results file not found: ./results/preliminary/preliminary_er4_N100_hetero_var_mean.csv

In [ ]:
# Results are displayed by run_or_load_preliminary_results.


## CASE 5 - N=100, ER4, Standardized

In [ ]:
N = 100
SCENARIO_NAME = 'er4_N100_var1_standardized'

n_dags = 50
verb = True
data_params = {
    'n_nodes': N,
    'n_samples': 1000, # 1000,
    'graph_type': 'er',
    'edges': 4*N,
    'edge_type': 'positive',
    'w_range': (.5, 1),  # (.5, 1)
    'var': 1
}
metrics, tables, exps_leg = run_or_load_preliminary_results(data_params, Exps_standardized, n_dags, SCENARIO_NAME, thr=.3, verb=verb)


## CASE 6 - N=100, ER2, Homocedastic

In [ ]:
N = 100
SCENARIO_NAME = 'er2_N100_var1'

n_dags = 50
verb = True
data_params = {
    'n_nodes': N,
    'n_samples': 1000, # 1000,
    'graph_type': 'er',
    'edges': 2*N,
    'edge_type': 'positive',
    'w_range': (.5, 1),  # (.5, 1)
    'var': 1
}
metrics, tables, exps_leg = run_or_load_preliminary_results(data_params, Exps, n_dags, SCENARIO_NAME, thr=.3, verb=verb)


Loaded mean results from ./results/preliminary/preliminary_er2_N100_var1_mean.csv


,leg,shd,fscore,sid_norm,err,acyc,time
0,NOMAD-adam,0.0400 ± 0.2800,0.9999 ± 0.0010,0.0164 ± 0.1148,0.0079 ± 0.0022,0.0134 ± 0.0172,36.5040 ± 8.0469
1,NOMAD-fista,0.1600 ± 0.4630,0.9994 ± 0.0019,0.1040 ± 0.3088,0.0068 ± 0.0021,0.0000 ± 0.0000,42.3520 ± 17.0778
2,NoTears,5.2800 ± 4.3637,0.9812 ± 0.0138,1.2560 ± 0.8431,0.0314 ± 0.0202,0.0000 ± 0.0000,221.6827 ± 128.9661
3,DAGMA,3.8000 ± 4.3955,0.9865 ± 0.0145,0.8394 ± 0.9706,0.0208 ± 0.0172,0.0001 ± 0.0000,41.4650 ± 10.8174
4,NonDAGMA,0.2400 ± 0.7889,0.9991 ± 0.0027,0.0584 ± 0.1871,0.0052 ± 0.0031,0.0001 ± 0.0000,48.1436 ± 11.6322
5,CoLiDE-EV,4.8200 ± 5.2448,0.9833 ± 0.0167,0.9068 ± 0.9393,0.0234 ± 0.0199,0.0001 ± 0.0000,46.5249 ± 14.3147
6,CoLiDE-NV,13.3800 ± 8.8451,0.9560 ± 0.0263,2.0518 ± 1.5403,0.0530 ± 0.0302,0.0000 ± 0.0000,51.6364 ± 16.4401
7,GOLEM-EV-np,8.3800 ± 6.8670,0.9698 ± 0.0225,1.9396 ± 1.5707,0.0406 ± 0.0309,0.0000 ± 0.0000,415.8982 ± 85.6386
8,GOLEM-EV,8.4000 ± 6.8673,0.9697 ± 0.0225,1.9396 ± 1.5707,0.0406 ± 0.0309,0.0000 ± 0.0000,1087.0660 ± 230.3706
9,GOLEM-NV,9.8400 ± 7.7443,0.9661 ± 0.0250,1.8428 ± 1.5398,0.0491 ± 0.0381,0.0000 ± 0.0000,1702.3634 ± 175.3862


Loaded median results from ./results/preliminary/preliminary_er2_N100_var1_median.csv


,leg,shd,fscore,sid_norm,err,acyc,time
0,NOMAD-adam,0.0000 ± 0.2800,1.0000 ± 0.0010,0.0000 ± 0.1148,0.0073 ± 0.0022,0.0070 ± 0.0172,37.2418 ± 8.0469
1,NOMAD-fista,0.0000 ± 0.4630,1.0000 ± 0.0019,0.0000 ± 0.3088,0.0064 ± 0.0021,0.0000 ± 0.0000,39.5240 ± 17.0778
2,NoTears,3.5000 ± 4.3637,0.9860 ± 0.0138,1.0700 ± 0.8431,0.0250 ± 0.0202,0.0000 ± 0.0000,201.7633 ± 128.9661
3,DAGMA,2.0000 ± 4.3955,0.9902 ± 0.0145,0.4850 ± 0.9706,0.0160 ± 0.0172,0.0001 ± 0.0000,42.6121 ± 10.8174
4,NonDAGMA,0.0000 ± 0.7889,1.0000 ± 0.0027,0.0000 ± 0.1871,0.0042 ± 0.0031,0.0001 ± 0.0000,48.9244 ± 11.6322
5,CoLiDE-EV,3.0000 ± 5.2448,0.9890 ± 0.0167,0.6200 ± 0.9393,0.0182 ± 0.0199,0.0001 ± 0.0000,45.3639 ± 14.3147
6,CoLiDE-NV,12.0000 ± 8.8451,0.9608 ± 0.0263,1.6000 ± 1.5403,0.0469 ± 0.0302,0.0000 ± 0.0000,50.8337 ± 16.4401
7,GOLEM-EV-np,6.0000 ± 6.8670,0.9766 ± 0.0225,1.5100 ± 1.5707,0.0299 ± 0.0309,0.0000 ± 0.0000,410.6770 ± 85.6386
8,GOLEM-EV,6.0000 ± 6.8673,0.9766 ± 0.0225,1.5100 ± 1.5707,0.0299 ± 0.0309,0.0000 ± 0.0000,1204.3185 ± 230.3706
9,GOLEM-NV,8.0000 ± 7.7443,0.9707 ± 0.0250,1.4250 ± 1.5398,0.0349 ± 0.0381,0.0000 ± 0.0000,1681.0109 ± 175.3862


In [ ]:
# Results are displayed by run_or_load_preliminary_results.
